# Hands-on: Run Hugging Face LLM Locally with transformers Pipeline

A step-by-step guide to running a Large Language Model locally using Hugging Face’s transformers pipeline.

##  Step 1: Install Required Libraries

We need to install the Hugging Face `transformers` library (for loading and running the LLM)  
and `accelerate` (for better hardware acceleration and device placement)

In [1]:
!pip install -q torch transformers

## Step 2: Import Libraries

Import the `pipeline` from Hugging Face `transformers` (for easy text generation)  
and `torch` (PyTorch) to handle tensor operations and device management.

In [2]:
from transformers import pipeline
import torch

## Step 3: Fill Hugging Face Token

Log in to Hugging Face Hub using your access token.  
This is required to download gated models or use private repositories.

In [3]:
# from huggingface_hub import login
# login(new_session=False)

## Step 4: Trying LLMs

### Google Gemma

Now, let's load the Google Gemma model (`google/gemma-3-1b-it`)  
using the `text-generation` pipeline from Hugging Face.

You can access the model here: https://huggingface.co/google/gemma-3-1b-it

#### Load the LLM (Google Gemma)

We also set the device automatically to:
- **GPU** (`cuda`) if available  
- otherwise, fallback to **CPU**

This will download the model weights and tokenizer files as shown below.

In [4]:
pipe = pipeline(
    "text-generation",
    model="google/gemma-3-1b-it",
    device="cuda" # if torch.cuda.is_available() else "cpu"
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-1b-it"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

#### Create Your Prompt

Define the input prompt you want the model to respond to.  
Here, we ask the assistant to "Write a poem on Hugging Face, the company".


In [6]:
prompt = "You are a helpful assistant.\nUser: Write a poem on Hugging Face, the company\nAssistant:"

#### Generate the Output

Use the pipeline to generate text based on your prompt.

- `max_new_tokens=50` → limit the response length  
- `temperature=0.7` → control creativity (higher is usually more creative)

Finally, print the generated text.

In [7]:
output = pipe(prompt,
              max_new_tokens=50,
              temperature=0.7)

print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant:

The patterns shift, a digital grace,
Hugging Face, a welcoming space.
With models vast, a learning tide,
For AI’s future, deep inside.

From transformers to datasets grand,
Innovation blooms, across the


In [8]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():  # disables grad to save memory
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant:
Okay, here's a poem about Hugging Face:

(Verse 1)
A neural network's dream, a vibrant hue,
Hugging Face, a place for all to view.
With models galore, a boundless


#### Try another prompt

In [9]:
prompt = "Explain clean energy in 2 sentences"

output = pipe(prompt, max_new_tokens=50)

print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain clean energy in 2 sentences.

Clean energy refers to energy that comes from sources that don't pollute the environment or deplete natural resources. It includes renewable sources like solar, wind, and hydro power, and aims to create a sustainable future for the planet.

---




### Meta LLama

#### Load the LLM (Llama)

In [10]:
pipe_llama = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

#### Create Prompt, Generate Output

In [12]:
prompt = 'The key to life is '
output = pipe_llama(prompt,
                    max_new_tokens=50)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The key to life is 3 simple words: 3 Words.
I’m sure you have heard of these three words. They are the ones that you can use to change your life. They are the ones that can help you get the life that you want. They are the


#### Try Another Prompt

In [13]:
prompt = 'What is the key to life?'
output = pipe_llama(prompt)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the key to life? Why do we exist? Where are we going? These are all questions that we can answer, and the answers can be found in the scriptures. The Book of Mormon is a wonderful book to help us understand why we are here, and what we should be doing while we are here.
The Book of Mormon is a book of many miracles, and it is a book that teaches us many things that we need to know. It is a book that teaches us about God, His creation, and our relationship with Him. It is a book that teaches us about the life, death, and resurrection of Jesus Christ. It is a book that teaches us about the plan of salvation, and how we can be saved.
The Book of Mormon is a book that teaches us about the importance of the gospel. It teaches us that we need to be humble, and that we need to be obedient to the commandments of God. It teaches us that we need to be grateful for the blessings that God has given us, and that we need to be willing to do whatever He asks of us. It teaches us that we need 

#### "Chat" Prompt

In [14]:
prompt = "You are a helpful assistant.\nUser: Write a poem on Hugging Face, the company\nAssistant:"
output = pipe_llama(prompt)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant: I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I am a helpful assistant. I 

`meta-llama/Llama-3.2-1B` is a powerful foundation model, it is not instruction-tuned — that's why the result is not as good as the instruction-tuned `google/gemma-1.1-3b-it` model

